# Training Loops and Loss Functions

> The previous chapter looked at MoE——a large model is not just a pile of Transformer Block, it may also contain a router, experts, auxiliary loss these training difficulties. a, sample Hugging Face Transformers, ModelScope ms-swift classindustrialtraining, finallybecomes loss parameterupdate.
>
> loss, a Trainer, finallyindustrialtrainingreal.

industrialtraining `trainer.train()`, modeltraining. :

```text
 / Conversation data
 ↓
Tokenizer Chat Template
 ↓
input_ids / attention_mask / labels
 ↓
Data Collator batch
 ↓
model(**batch) get logits loss
 ↓
loss.backward()
 ↓
optimizer.step()
 ↓
lr_scheduler.step()
 ↓
save checkpoint / / evaluation
```

head.

Note:

| | training | |
|:---|:---|:---|
| | Hugging Face Transformers | `Trainer`, `TrainingArguments`, `DataCollatorForLanguageModeling` |
| / ModelScope | ModelScope SWIFT, called `ms-swift` | `swift sft`, `SftArguments`, LoRA/QLoRA/DeepSpeed |

`ms-swift` ModelScope modeltraining, modelload, data, Chat Template, SFT, LoRA, distributedtraining, benchmark. Transformer , training loop . 

In [ ]:
# All imports for this chapter live in the import ; this chapter uses code cell
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader

torch.manual_seed(42)
print("This chapter uses PyTorch to mimic the core logic behind industrial training libraries. ")
print("Key observation: Trainer It is not magic, it just packages data processing, forward, loss, backward, step . ")

## 0. Build Intuition First: What Does a Trainer Actually Save You?

First define three terms.

Trainer A trainer is a training manager, it is not a model itself, but a program that repeats the training steps. Hugging Face `Trainer` `ms-swift` SFT class.

Batch modelbatchsample. GPU 2 conversation, length 8 token, batch `input_ids` shapeis exactly `[2, 8]`.

Loss model. languagemodel Cross-Entropy Loss: correct token probability, loss .

training loop , orderexecutestep:

```text
1. dataset batchsample
2. tokenizer token id
3. data collator manysample pad a batch
4. model forward, get logits
5. labels cross-entropy loss
6. loss.backward() gradients
7. optimizer.step() updateparameter
8. , evaluation, save checkpoint
```

PyTorch training loop, 7 10 . industrialtrainingso, becauserealtraining 7 , VRAM, , gradients, distributed, resume point, evaluation, saveweight, LoRA .

training loop : model input_ids, predict labels, loss , updateparameter. 

## 1. How Does a Text Sample Become a Training Sample?

We avoid real Tokenizer, and hand-build a tinyvocabulary. so every number is easy to inspect.

```text
vocabulary:
0=<pad>, 1=<bos>, 2=<eos>, 3=, 4=, 5=machine, 6=learning

: machine learning
complete token sequence: [<bos>, , , machine, learning, <eos>]
: [1, 3, 4, 5, 6, 2]
```

languagemodeltraining, “answer”. answerInnext token :

```text
input_ids = [1, 3, 4, 5, 6]
labels = [3, 4, 5, 6, 2]
```

is exactly:

```text
see <bos> → should predict 
see <bos> → should predict 
see <bos> → should predict machine
see <bos> machine → should predict learning
see <bos> machine learning → should predict <eos>
```

called **next-token prediction**: token, predictnext token. 

In [ ]:
# Build input_ids labels
id_to_token = {
 0: "<pad>",
 1: "<bos>",
 2: "<eos>",
 3: "",
 4: "",
 5: "machine",
 6: "learning",
}

sentence = torch.tensor([1, 3, 4, 5, 6, 2])
input_ids = sentence[:-1]
labels = sentence[1:]

print("complete:", sentence.tolist())
print("input_ids:", input_ids.tolist())
print("labels: ", labels.tolist())
print()

for pos in range(len(input_ids)):
 context = [id_to_token[i.item()] for i in input_ids[:pos + 1]]
 target = id_to_token[labels[pos].item()]
 print(f"positions {pos}: see {context} -> predict {target}")

print()
print("Key observation: labels answer, /get. ")

## 2. Why Does the Chat Template Affect Loss?

plainPre-trainingdata is a continuous text, every token position contributes loss loss. but SFT (supervisedFine-tuning) Conversation data:

```text
user: 2 + 2 ？
assistant: 4
```

a: model？

modellearning **assistant **, because user part. industrialtraining label :

```text
input_ids: [<user>, 2, +, 2, ?, <assistant>, 4, <eos>]
labels: [-100, -100, -100, -100, -100, -100, 4, <eos>]
```

 `-100` PyTorch `cross_entropy` default. : positions loss.

**Chat Template**: structureconversationconvert tomodel. `user` `assistant` . differentmodeldifferent, Qwen, LLaMA, ChatGLM .

soIn `ms-swift` Transformers , SFT data“tokenize”, :

```text
messages -> apply_chat_template -> tokenize -> labels mask
```

In [ ]:
# Simulate Chat Template label mask
vocab = {
 "<pad>": 0,
 "<user>": 1,
 "<assistant>": 2,
 "<eos>": 3,
 "2": 4,
 "+": 5,
 "=": 6,
 "?": 7,
 "4": 8,
}

chat_tokens = ["<user>", "2", "+", "2", "?", "<assistant>", "4", "<eos>"]
input_ids = torch.tensor([vocab[token] for token in chat_tokens])

# training only the assistant assistant reply: positions 6 7 loss, the restpositions
labels = torch.tensor([-100, -100, -100, -100, -100, -100, vocab["4"], vocab["<eos>"]])

print("tokens: ", chat_tokens)
print("input_ids:", input_ids.tolist())
print("labels: ", labels.tolist())
print()

for token, label in zip(chat_tokens, labels.tolist()):
 if label == -100:
 print(f"{token:>11s} -> loss")
 else:
 print(f"{token:>11s} -> loss, goal token id = {label}")

print()
print("Key observation: SFT model user, assistant assistant replypart. ")

## 3. Cross-Entropy by Hand: How Is Loss Computed?

modelIneachpositionsoutputa token, but a full row of scores called, called **logits**.

**Logits**: the model's raw scores forvocabularyeach token , probability. vocabulary 4 token, positions logits :

```text
[2.0, 1.0, 0.1, -1.0]
```

 loss, softmax:

$$p_i = \frac{e^{z_i}}{\sum_j e^{z_j}}$$

thencorrect token probability $p_{correct}$:

$$loss = -\log(p_{correct})$$

 log？:

```text
correct probability = 1.0 -> -log(1.0) = 0 
correct probability = 0.5 -> -log(0.5) = 0.693 
correct probability = 0.01 -> -log(0.01) = 4.605 
```

In [ ]:
# apositions Cross-Entropy Loss
logits = torch.tensor([2.0, 1.0, 0.1, -1.0])
correct_id = 0

exp_values = torch.exp(logits)
probs = exp_values / exp_values.sum()
correct_prob = probs[correct_id].item()
manual_loss = -math.log(correct_prob)

torch_loss = F.cross_entropy(logits.view(1, -1), torch.tensor([correct_id]))

print("logits:", logits.tolist())
print("exp(logits):", [round(x, 4) for x in exp_values.tolist()])
print("softmax probability:", [round(x, 4) for x in probs.tolist()])
print()
print(f"correct token id = {correct_id}")
print(f"correct token probability = {correct_prob:.4f}")
print(f" loss = -log({correct_prob:.4f}) = {manual_loss:.4f}")
print(f"PyTorch loss = {torch_loss.item():.4f}")
print()
print("Key observation: cross_entropy = log_softmax + correctclass + . ")

## 4. What Happens When a Batch Enters the Model?

Industrial trainers do not train one sample at a time, they pack manysampletogether into a batch.

Here comes the question: what if two sentences differ in length?？

```text
sample A: [<bos>, , , machine, learning, <eos>] length 6
sample B: [<bos>, , , learning, <eos>] length 5
```

atensor, pad :

```text
input_ids:
A: [1, 3, 4, 5, 6]
B: [1, 3, 4, 6, 0]

labels:
A: [3, 4, 5, 6, 2]
B: [3, 4, 6, 2, -100]
```

`attention_mask` modelpositions token, pad:

```text
A: [1, 1, 1, 1, 1]
B: [1, 1, 1, 1, 0]
```

**Data Collator**: many tokenize samplea batch . padding, generate `attention_mask`, `labels`. 

In [ ]:
# Write a minimal Data Collator, simulating how industrial trainers pack batch process
def simple_collate(features, pad_id=0, ignore_index=-100):
 """
 manysamplepad into a batch.

 parameter:
 features: sample input_ids labels
 pad_id: input_ids use padding token id
 ignore_index: labels use

 returns:
 a batch , input_ids, attention_mask, labels
 """
 max_len = max(len(item["input_ids"]) for item in features)

 batch_input_ids = []
 batch_attention_mask = []
 batch_labels = []

 for item in features:
 input_ids = item["input_ids"]
 labels = item["labels"]
 pad_len = max_len - len(input_ids)

 batch_input_ids.append(input_ids + [pad_id] * pad_len)
 batch_attention_mask.append([1] * len(input_ids) + [0] * pad_len)
 batch_labels.append(labels + [ignore_index] * pad_len)

 return {
 "input_ids": torch.tensor(batch_input_ids),
 "attention_mask": torch.tensor(batch_attention_mask),
 "labels": torch.tensor(batch_labels),
 }

features = [
 {"input_ids": [1, 3, 4, 5, 6], "labels": [3, 4, 5, 6, 2]},
 {"input_ids": [1, 3, 4, 6], "labels": [3, 4, 6, 2]},
]

batch = simple_collate(features)

for name, value in batch.items():
 print(f"{name}: shape={tuple(value.shape)}")
 print(value)
 print()

print("Key observation: input_ids pad_id , labels -100 . ")
print("if labels pad_id, modellearningpredict <pad>, wrong. ")

## 5. Model Forward: Why Can We Pass labels Straight to the Model?

In Transformers the common pattern is:

```python
outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
loss = outputs.loss
logits = outputs.logits
```

This does not mean the model“needs the answer to predict”. the real order is:

```text
1. the model uses input_ids forward, get logits
2. if labels, model cross_entropy loss
3. returns outputs.loss outputs.logits
```

is exactly, `labels` loss, will notmodelpredict.

a Causal LM. Transformer Block, Embedding + Linear . model, trainingmodelshape loss computeway. 

In [ ]:
class TinyCausalLM(nn.Module):
 """
 a Causal Language Model, Trainer model.

 parameter:
 vocab_size: vocabularysize
 hidden_size: token dimension
 """
 def __init__(self, vocab_size, hidden_size):
 super().__init__()
 self.embedding = nn.Embedding(vocab_size, hidden_size)
 self.lm_head = nn.Linear(hidden_size, vocab_size)

 def forward(self, input_ids, attention_mask=None, labels=None):
 """
 input token ids, output logits; if labels, at the same timecompute loss.

 parameter:
 input_ids: [batch, seq_len]
 attention_mask: [batch, seq_len], compute
 labels: [batch, seq_len], -100 positions loss

 returns:
 a, loss logits
 """
 hidden = self.embedding(input_ids)
 logits = self.lm_head(hidden)

 loss = None
 if labels is not None:
 loss = F.cross_entropy(
 logits.reshape(-1, logits.size(-1)),
 labels.reshape(-1),
 ignore_index=-100,
 )

 return {"loss": loss, "logits": logits}

model = TinyCausalLM(vocab_size=9, hidden_size=16)
outputs = model(**batch)

print("logits shape:", tuple(outputs["logits"].shape))
print("loss:", round(outputs["loss"].item(), 4))
print()
print("Key observation: logits [batch, seq_len, vocab_size]. ")
print("loss every -100 positionsaverage. ")

## 6. A Minimal Trainer: The Skeleton of Industrial Libraries

In pieces . atraining step :

```text
batch = next(dataloader)
outputs = model(**batch)
loss = outputs.loss
loss.backward()
optimizer.step()
optimizer.zero_grad()
```

butindustrialtraining:

| | need |
|:---|:---|
| gradient accumulation | VRAM batch, batch gradients |
| lr scheduler | learning ratecannot, warmup |
| mixed precision | FP16/BF16 accelerationVRAM |
| gradient clipping | gradients |
| checkpoint | trainingresume |
| eval / logging | model |
| distributed training | GPU / machineparalleltraining |

adistributed. `Trainer` difference. 

In [ ]:
class ToyTextDataset(Dataset):
 """
 adata, sample token ids.

 parameter:
 sequences: complete token sequence, BOS/EOS
 """
 def __init__(self, sequences):
 self.sequences = sequences

 def __len__(self):
 return len(self.sequences)

 def __getitem__(self, index):
 ids = self.sequences[index]
 return {
 "input_ids": ids[:-1],
 "labels": ids[1:],
 }

sequences = [
 [1, 3, 4, 5, 6, 2],
 [1, 3, 4, 6, 2],
 [1, 3, 4, 5, 2],
 [1, 3, 4, 6, 2],
]

dataset = ToyTextDataset(sequences)
dataloader = DataLoader(dataset, batch_size=2, shuffle=False, collate_fn=simple_collate)

model = TinyCausalLM(vocab_size=9, hidden_size=32)
optimizer = torch.optim.AdamW(model.parameters(), lr=0.05)

print("training, a batch: ")
first_batch = next(iter(dataloader))
for name, value in first_batch.items():
 print(name, tuple(value.shape))
print()
print("Key observation: Dataset sample, DataLoader + collator batch. ")

In [ ]:
# training loop: is exactly Trainer part
num_epochs = 8
log_every = 1

for epoch in range(num_epochs):
 total_loss = 0.0
 num_steps = 0

 for batch in dataloader:
 outputs = model(**batch)
 loss = outputs["loss"]

 loss.backward()
 optimizer.step()
 optimizer.zero_grad()

 total_loss += loss.item()
 num_steps += 1

 avg_loss = total_loss / num_steps
 if (epoch + 1) % log_every == 0:
 print(f"epoch {epoch + 1:02d} | train_loss = {avg_loss:.4f}")

print()
print("Key observation: loss because Trainer , parameter optimizer . ")

## 7. The Optimizer: How Do Parameters Update After Gradients?

> atraining loop, `optimizer.step()`——modelparameterInupdate. but step , AdamW SGD, .
>
> Adam , optimizer.step() : , , bias correction, weight decay, gradient clipping, finally 2024 Muon. , "eachoptimizerIn".

optimizer: gradients, parameter, . SGD $\theta \leftarrow \theta - \eta g$ update, butInmodelparameter, training, SGD ——gradients, differentparameter, update.

Adam average, eachparameteralearning rate, LLM training. Adam ——2024 Keller Jordan In nanoGPT Speedrun Muon training, optimizer (Shampoo, SOAP) . , need Adam . 

### 7.1 Adam: + 

Adam eachparameter: $m$ (gradientsaverage, called momentum) $v$ (gradientsaverage) . In $t$ , gradients $g_t$ :

$$m_t = \beta_1 m_{t-1} + (1-\beta_1) g_t$$
$$v_t = \beta_2 v_{t-1} + (1-\beta_2) g_t^2$$

$m$ : ifgradients, $m$ , ; ifgradients, , $m$ 0. In noisy gradient .

$v$ : eachparameterpositions $v$, update $\sqrt{v} + \epsilon$. gradientspositions, $v$ , updateautomatic——learning rate.

 0 initialize, so. Adam bias correction :

$$\hat{m}_t = \frac{m_t}{1 - \beta_1^t}, \quad \hat{v}_t = \frac{v_t}{1 - \beta_2^t}$$

 $t$ $\beta^t \to 0$, 1, Invalue. update:

$$\theta_t = \theta_{t-1} - \eta \cdot \frac{\hat{m}_t}{\sqrt{\hat{v}_t} + \epsilon}$$

: $\beta_1 = 0.9$, $\beta_2 = 0.999$, $\epsilon = 10^{-8}$. LLM Pre-traininglearning rateIn $10^{-4}$ . $\beta_2$ 0.999 becausegradients, need. 

#### a Adam step

parametervalue $\theta_0 = 1.0$, gradients $g_1 = 0.5$, $\beta_1 = 0.9$, $\beta_2 = 0.999$, $\eta = 0.1$, $\epsilon = 10^{-8}$. $m_0 = v_0 = 0$.

- $m_1 = 0.9 \cdot 0 + 0.1 \cdot 0.5 = 0.05$
- $v_1 = 0.999 \cdot 0 + 0.001 \cdot 0.25 = 0.00025$
- $\hat{m}_1 = 0.05 / (1 - 0.9) = 0.5$ ( $g_1$, bias correction resume)
- $\hat{v}_1 = 0.00025 / (1 - 0.999) = 0.25$ ( $g_1^2$)
- $\Delta\theta = -0.1 \cdot 0.5 / (\sqrt{0.25} + 10^{-8}) = -0.1$
- $\theta_1 = 1.0 - 0.1 = 0.9$

Key observation: bias correction update $\eta \cdot \text{sign}(g_1)$, $m_1 = 0.05$ , $v_1 = 0.00025$ , update. 

In [ ]:
import torch

torch.manual_seed(42)

# verify: parameter Adam
theta = torch.tensor([1.0], requires_grad=True)
theta.grad = torch.tensor([0.5])

optimizer = torch.optim.Adam([theta], lr=0.1, betas=(0.9, 0.999), eps=1e-8)
optimizer.step()

print(f"theta update: 1.0")
print(f"theta update: {theta.item():.6f}")
print(f": 0.9")
print(f": {abs(theta.item() - 0.9):.2e}")
print()
print("Key observation: bias correction Adam update lr * sign(grad). ")
print("if correction, update. ")

#### SimpleAdam

 Adam waya. `torch.optim.Optimizer`, In `step()` update. PyTorch , AMP, foreach optimize. 

In [ ]:
import torch

class SimpleAdam(torch.optim.Optimizer):
 """ Adam, , """

 def __init__(self, params, lr=1e-3, betas=(0.9, 0.999), eps=1e-8):
 defaults = {"lr": lr, "betas": betas, "eps": eps}
 super().__init__(params, defaults)

 def step(self):
 for group in self.param_groups:
 lr = group["lr"]
 beta1, beta2 = group["betas"]
 eps = group["eps"]
 for p in group["params"]:
 if p.grad is None:
 continue
 grad = p.grad.data
 state = self.state[p]

 # initialize
 if "t" not in state:
 state["t"] = 0
 state["m"] = torch.zeros_like(p.data)
 state["v"] = torch.zeros_like(p.data)

 state["t"] += 1
 t = state["t"]
 m, v = state["m"], state["v"]

 # average
 m.mul_(beta1).add_(grad, alpha=1 - beta1)
 v.mul_(beta2).addcmul_(grad, grad, value=1 - beta2)

 # bias correction
 m_hat = m / (1 - beta1 ** t)
 v_hat = v / (1 - beta2 ** t)

 # parameterupdate
 p.data.sub_(lr * m_hat / (v_hat.sqrt() + eps))

# verify: PyTorch Adam Ininputoutput
torch.manual_seed(0)
p1 = torch.randn(100, requires_grad=True)
p2 = p1.clone().detach().requires_grad_(True)
grad = torch.randn(100) * 0.1
p1.grad = grad.clone()
p2.grad = grad.clone()

opt_torch = torch.optim.Adam([p1], lr=1e-3)
opt_ours = SimpleAdam([p2], lr=1e-3)
opt_torch.step()
opt_ours.step()

max_diff = (p1 - p2).abs().max().item()
print(f"PyTorch Adam vs SimpleAdam : {max_diff:.2e}")

# 99 
for _ in range(99):
 g = torch.randn(100) * 0.1
 p1.grad = g.clone()
 p2.grad = g.clone()
 opt_torch.step()
 opt_ours.step()

max_diff = (p1 - p2).abs().max().item()
print(f"100 : {max_diff:.2e}")
print("Key observation: Adam value, correct. ")

### 7.2 AdamW: weight decay

Adam if weight decay (L2 ) , $\lambda \theta$ gradients: $g_t^{\text{reg}} = g_t + \lambda \theta$, thengradients $m$, $v$ update.

In, L2 penalty gradients $\lambda \theta$ $m / \sqrt{v}$ ——weight decay learning ratescaling. gradientsparameter, $\sqrt{v}$ , weight decay actual; gradientsparameter, weight decay . weight decay"parameter 0 ".

AdamW (Loshchilov & Hutter, 2019) weight decay gradients: Inparameter, $m / \sqrt{v}$. updatebecomes:

$$\theta_t = \theta_{t-1} - \eta \cdot \frac{\hat{m}_t}{\sqrt{\hat{v}_t} + \epsilon} - \eta \lambda \theta_{t-1}$$

 decay In Adam update (`p.data -= lr * weight_decay * p.data`) , $\hat{m}/\sqrt{\hat{v}}$ parameter. 

#### parameter decay, decay

 recipe: bias LayerNorm parameter weight decay, (Linear weight, Embedding) . reason bias LayerNorm activationvaluevalue, 0 . Linear weight Embedding scalingparameter, weight decay valueactivation.

PyTorch parameter group : `p.dim() >= 2` a——2D tensor Linear/Conv/Embedding weight, 1D bias LayerNorm $\gamma, \beta$. 

In [ ]:
import torch
import torch.nn as nn

# a decoder-only model, parameter
class TinyBlock(nn.Module):
 def __init__(self, d):
 super().__init__()
 self.ln = nn.LayerNorm(d)
 self.fc = nn.Linear(d, d * 4)
 self.proj = nn.Linear(d * 4, d)

 def forward(self, x):
 return self.proj(torch.nn.functional.gelu(self.fc(self.ln(x))))

model = nn.Sequential(
 nn.Embedding(1000, 64),
 TinyBlock(64),
 TinyBlock(64),
 nn.Linear(64, 1000),
)

# dimension: 2D decay, 1D decay
decay, no_decay = [], []
for name, p in model.named_parameters():
 if not p.requires_grad:
 continue
 if p.dim() >= 2:
 decay.append(p)
 else:
 no_decay.append(p)

print(f"decay parametercount: {len(decay)}, : {sum(p.numel() for p in decay)}")
print(f"no_decay parametercount: {len(no_decay)}, : {sum(p.numel() for p in no_decay)}")
print()
print(": decay (Linear/Embedding weight) , no_decay bias LayerNorm. ")

### 7.3 Gradient Clipping

training, modelparameter, loss surface predict. outlier batch (data, sequence) gradients, parameter. Gradient clipping gradients, update.

LLM global norm clip: computeeveryparametergradients global L2 norm $\|g\| = \sqrt{\sum_i \|g_i\|^2}$, ifvalue $\tau$, everygradients $\tau / \|g\|$ :

$$g \leftarrow g \cdot \min\left(1, \frac{\tau}{\|g\|}\right)$$

parametergradients, scaling. value $\tau$ In GPT-3, LLaMA model 1.0.

Transformer need clip: attention softmax , activation head gradientsaveragevaluecount; layer. clip, bad batch layerparameter. 

In [ ]:
import torch

def clip_grad_norm_(params, max_norm, eps=1e-6):
 """ global norm clip, equivalent torch.nn.utils.clip_grad_norm_

 parameter:
 params: iterationparameter (grad cannot None)
 max_norm: gradients global norm 
 eps: 0 
 """
 grads = [p.grad for p in params if p.grad is not None]
 if len(grads) == 0:
 return torch.tensor(0.0)

 # compute global norm: everygradients L2 norm 
 total_norm_sq = sum(g.float().pow(2).sum() for g in grads)
 total_norm = total_norm_sq.sqrt()

 # ifvalue, everygradients
 clip_coef = max_norm / (total_norm + eps)
 if clip_coef < 1:
 for g in grads:
 g.mul_(clip_coef)

 return total_norm

# verify: constructgradients, global norm 1.0
params = [torch.randn(100, requires_grad=True) for _ in range(3)]
for p in params:
 p.grad = torch.randn(100) * 10 # gradients

before_norm = torch.sqrt(sum(p.grad.float().pow(2).sum() for p in params)).item()
print(f"clip global norm: {before_norm:.2f}")

returned = clip_grad_norm_(params, max_norm=1.0)
after_norm = torch.sqrt(sum(p.grad.float().pow(2).sum() for p in params)).item()
print(f"clip global norm: {after_norm:.4f}")
print()
print("Key observation: clip global norm 1.0 , butgradients. ")
print("clip will notgradients—— norm value, clip_coef >= 1 but. ")

### 7.4 Muon: matrixorthogonalizationgradientsupdate

Muon (Keller Jordan, 2024) observe: Linear layerweighta 2D matrix $W \in \mathbb{R}^{m \times n}$, correspondinggradients $G$ 2D. Adam $G$ , $m/\sqrt{v}$, matrixstructure. if $G$ matrix, can Newton-Schulz iterationorthogonalization, geta, update.

orthogonalization: gradients $G$ SVD $G = U \Sigma V^\top$, orthogonalizationis exactlyvalue 1, get $U V^\top$. update, ——class sign-gradient, butmatrixevery.

 SVD . Newton-Schulz iterationaorthogonalizationiteration:

$$X_{k+1} = \frac{1}{2} X_k (3 I - X_k^\top X_k)$$

 $X_0 = G / \|G\|_F$ , iteration 7-10 $X_k$ $G$ orthogonalization $U V^\top$. iterationneedmatrix, SVD acount.

Muon 2D matrixparameter. 1D parameter (bias, LayerNorm) matrixstructure, orthogonalization, Adam. soactualuse"Muon for 2D + Adam for 1D"optimizer. 

In [ ]:
import torch

torch.manual_seed(42)

def newton_schulz(G, steps=7):
 """ Newton-Schulz iteration G orthogonalization

 parameter:
 G: 2D gradientsmatrix
 steps: iteration, 7-10 
 returns:
 orthogonalizationmatrix, shape G 
 """
 assert G.dim() == 2, "Newton-Schulz 2D matrix"
 X = G.bfloat16() if G.is_cuda else G.float()
 # : < 1.5, iteration
 X = X / (X.norm() + 1e-7)
 for _ in range(steps):
 A = X @ X.t()
 B = A @ X
 X = 1.5 * X - 0.5 * B
 return X

# verify Newton-Schulz orthogonalization
G = torch.randn(8, 16)
X = newton_schulz(G, steps=7)

# orthogonalization: X X^T matrix
gram = X @ X.t()
identity = torch.eye(8)
off_diag = (gram - identity).abs().max().item()
diag_dev = (gram.diag() - 1.0).abs().max().item()

print(f"inputmatrix G shape: {tuple(G.shape)}")
print(f"iteration 7 X X^T 1: {diag_dev:.2e}")
print(f"iteration 7 X X^T value: {off_diag:.2e}")
print()
print("Key observation: Newton-Schulz iteration 7 , X X^T matrix. ")
print("equivalent G SVD value 1, but. ")

#### Muon optimizer

 Newton-Schulz `step()`, momentum, is exactlya Muon. RMS scaling, parametershape, butcomplete. Muon In [KellerJordan/Muon](https://github.com/KellerJordan/Muon), nanoGPT Speedrun Muon training 45 22 .

Muon learning rate Adam acount ($4 \times 10^{-3}$ vs $3 \times 10^{-4}$) , becauseorthogonalizationupdate, can. 

In [ ]:
import torch

class SimpleMuon(torch.optim.Optimizer):
 """ Muon: 2D parameter Newton-Schulz orthogonalization, 1D momentum SGD

 parameter:
 params: optimizeparameter
 lr: learning rate (Muon 4e-3, Adam acount)
 momentum: 
 ns_steps: Newton-Schulz iteration
 """

 def __init__(self, params, lr=4e-3, momentum=0.95, ns_steps=5):
 defaults = {"lr": lr, "momentum": momentum, "ns_steps": ns_steps}
 super().__init__(params, defaults)

 def step(self):
 for group in self.param_groups:
 lr = group["lr"]
 momentum = group["momentum"]
 ns_steps = group["ns_steps"]
 for p in group["params"]:
 if p.grad is None:
 continue
 grad = p.grad
 state = self.state[p]

 if len(state) == 0:
 state["momentum_buffer"] = torch.zeros_like(p)

 buf = state["momentum_buffer"]
 buf.mul_(momentum).add_(grad)

 if p.dim() >= 2:
 # 2D parameter: Newton-Schulz orthogonalization momentum
 update = newton_schulz(buf, steps=ns_steps)
 # sqrt(max(m, n)) scaling, Muon 
 scale = max(1.0, buf.shape[0] / buf.shape[1]) ** 0.5
 p.data.add_(update, alpha=-lr * scale)
 else:
 # 1D parameter: plain momentum SGD
 p.data.add_(buf, alpha=-lr)

# : Muon , parameter
torch.manual_seed(0)
p = torch.randn(32, 64, requires_grad=True)
p.grad = torch.randn(32, 64) * 0.1
before = p.data.clone()
opt = SimpleMuon([p], lr=4e-3, ns_steps=7)
opt.step()
delta = (p - before).abs().mean().item()
print(f"parameteraverageupdate: {delta:.6f}")
print(f"parameterdimension: {tuple(p.shape)}")
print("Key observation: Muon 2D parameter Newton-Schulz orthogonalizationupdate. ")

### 7.5 Lion / Shampoo / SOAP 

Adam Muon , 2023-2025 valueoptimizer. : Adam eachparameterpositions, gradientsmatrix/tensorstructure.

**Lion** (Google, 2023) update: $\theta \leftarrow \theta - \eta \cdot \text{sign}(m)$, $m$ momentum. Adam difference Lion , sign——equivalentalearning rate. Lion $v$ , eachparameterVRAM; In Vision LLM AdamW result. learning rate weight decay .

**Shampoo** (Gupta et al., 2018) optimizer. matrix $L = \mathbb{E}[g g^\top]$ () $R = \mathbb{E}[g^\top g]$ () , $L^{-1/4} G R^{-1/4}$ update. Adam Inmatrix——Adam matrix, Shampoo completematrix. $L$, $R$ , Inmodeluse.

**SOAP** (Shi et al., 2024) Shampoo optimize. $L$, $R$ updateIn ( N update) , Adam . SOAP In LLaMA scale AdamW 30%-50% .

optimizer Adam/Muon differenceIngradientsmatrixstructure:

| optimizer | matrixstructure | | | |
|:---|:---|:---|:---|:---|
| Adam/AdamW | () | 2× parameter | | |
| Lion | (sign) | 1× parameter | | |
| Muon | (orthogonalization) | 1× parameter | Adam | |
| Shampoo | (complete) | matrix | | |
| SOAP | (+) | | Adam 30-50% | |

aseeoptimizer. AdamW GPT-3, LLaMA, Qwen, DeepSeek modelverify, . Muon SOAP needlearning rate, weight decay, warmup, . , AdamW + LR schedule . 

### 7.6 optimizersummary

differenttrainingoptimizer:

| training | optimizer | learning rate | |
|:---|:---|:---|:---|
| Pre-training (dense) | AdamW | $3 \times 10^{-4}$ | , |
| Pre-training (MoE) | AdamW | $3 \times 10^{-4}$ | router lr |
| SFT / Fine-tuning | AdamW | $10^{-5}$ $10^{-4}$ | weight decay 0 |
| RLHF (PPO) | AdamW | actor | actor/critic different lr |
| scale | Muon | $4 \times 10^{-3}$ | 2D Muon, 1D Adam |

optimizerlearning rate——Adam lr cannot Muon , . 

## 8. Mapping the Minimal Trainer to Hugging Face Transformers

if Transformers, :

```python
from transformers import AutoModelForCausalLM, AutoTokenizer
from transformers import DataCollatorForLanguageModeling
from transformers import Trainer, TrainingArguments

model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen2.5-0.5B")
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-0.5B")

collator = DataCollatorForLanguageModeling(
 tokenizer=tokenizer,
 mlm=False, # Causal LM, BERT masked LM
)

args = TrainingArguments(
 output_dir="outputs/qwen-sft",
 per_device_train_batch_size=2,
 gradient_accumulation_steps=8,
 learning_rate=2e-5,
 num_train_epochs=3,
 logging_steps=10,
 save_steps=500,
)

trainer = Trainer(
 model=model,
 args=args,
 train_dataset=train_dataset,
 data_collator=collator,
)

trainer.train()
```

 loop corresponding:

| | Transformers |
|:---|:---|
| `ToyTextDataset` | `Dataset` / `datasets.Dataset` |
| `simple_collate` | `DataCollatorForLanguageModeling` collator |
| `TinyCausalLM` | `AutoModelForCausalLM` |
| `optimizer` | Trainer create AdamW optimizer |
| `for epoch... for batch...` | `trainer.train()` |
| `outputs = model(**batch)` | Trainer `training_step` |
| `loss.backward()` | Trainer / Accelerate backward |
| `optimizer.step()` | Trainer gradients step |

Notea: Pre-training (CPT) conversation SFT collator .

| trainingtype | labels |
|:---|:---|
| Pre-training / Causal LM | every PAD token loss |
| Fine-tuning / SFT | user/system part `-100`, training assistant assistant reply |

## 9. Mapping the Same Pipeline to ModelScope ms-swift

 “ms-xxx” is exactly **ModelScope SWIFT / `ms-swift`**.

can: modelmodeltraining. need `Trainer`, LoRA , , DeepSpeed configbenchmark, automatic.

class:

```bash
swift sft \
 --model Qwen/Qwen2.5-0.5B-Instruct \
 --dataset your_dataset.jsonl \
 --train_type lora \
 --learning_rate 2e-5 \
 --per_device_train_batch_size 2 \
 --gradient_accumulation_steps 8 \
 --num_train_epochs 3 \
 --output_dir output/qwen-lora
```

, loop:

| `ms-swift` parameter/ | training |
|:---|:---|
| `--model` | loadmodel tokenizer |
| `--dataset` | readsample, convert totraining dataset |
| model template | messages convert tomodelneed Chat Template |
| `--train_type lora` | partparametergradients |
| `--per_device_train_batch_size` | sample |
| `--gradient_accumulation_steps` | batch update |
| `--learning_rate` | optimizer |
| `--deepspeed` / distributedparameter | multi-GPUmodel, gradients, optimizer |
| `--output_dir` | save checkpoint |

Inuse, because ModelScope model, data, Qwen , trainingconfig.

but: `ms-swift` a“training”. . :

```text
batch -> model forward -> loss -> backward -> optimizer step
```

## 10. The Full Pipeline of an Industrial Training Loop

content:

```text
1. data
 : "machinelearning"
 conversation: [{role: user, content: ...}, {role: assistant, content: ...}]

2. 
 apply_chat_template(messages)
 -> "<user>...<assistant>..."

3. Tokenize
 tokenizer(text)
 -> input_ids

4. construct labels
 Pre-training: labels ≈ input_ids next token
 SFT: user/system positions -100, assistant positionsanswer

5. Data Collator
 sample pad length
 -> input_ids / attention_mask / labels

6. Forward
 outputs = model(**batch)
 -> logits: [batch, seq_len, vocab_size]
 -> loss: token average cross-entropy

7. Backward
 loss.backward()
 -> eachtrainingparametergetgradients

8. Optimizer Step
 AdamW / 8-bit Adam / optimizer
 -> updateparameter

9. Scheduler / Logging / Eval / Checkpoint
 learning rate, loss, verify, saveweight
```

if LoRA, :

```text
modelpartparameter frozen, training only the LoRA A/B matrix.
```

if DeepSpeed ZeRO / FSDP, :

```text
parameter, gradients, optimizer GPU , but loss/backward/step .
```

## 11. Loss Through Information Theory: Perplexity, Entropy, and KL

 CE , "correct token probability". buttraining: perplexity (PPL) , entropy, KL divergence. CE , use.

: perplexity CE , CE / entropy / KL , finallyIn, RLHF, DPO, MoE load balancing . 

### 11.1 Perplexity: loss "modelIn token "

Cross-Entropy loss a, value. Perplexity (PPL) CE :

$$ \text{PPL} = \exp(\text{CE}) $$

PPL can: modelIneachpositions, equivalentIn token .

:

| CE loss | PPL | |
|:---|:---|:---|
| 0 | 1.0 | modelcorrect token, |
| 2.0 | 7.39 | modelIneachpositionsaverageIn 7-8 token |
| 4.6 | 100 | range 100 token |
| $\log V$ | $V$ | the model's raw scores forvocabularyeach token , equivalent |

 $V$ vocabularysize. PPL = vocabularysizemodelpredict, PPL .

a: PPL data. vocabulary, token , different, CE value, compare PPL valuewrong. PPL , datamodel. 

In [ ]:
# CE PPL 
import math

ce_values = [0.0, 1.0, 2.0, 3.0, 4.6, math.log(50000)]

print(f"{'CE':>6} | {'PPL':>12} | ")
print("-" * 50)
for ce in ce_values:
 ppl = math.exp(ce)
 if ce == 0.0:
 note = ", "
 elif ce == math.log(50000):
 note = f"In 50000 vocabulary"
 else:
 note = f"averageIn {ppl:.1f} token "
 print(f"{ce:>6.2f} | {ppl:>12.2f} | {note}")

print()
print("Key observation: PPL loss , . ")
print("but PPL data, Ina tokenizer . ")

### 11.2 CE, KL entropy 

Cross-Entropy, KL divergence entropy a:

$$ H(p, q) = H(p) + D_{KL}(p \| q) $$

 $p$ real, $q$ modelpredict. $H(p, q)$ cross-entropy, $H(p)$ real entropy, $D_{KL}(p \| q)$ KL , .

In teacher forcing languagemodeltraining, "real" one-hot: correct token probability 1, token probability 0. one-hot entropy 0:

$$ H(p) = -\sum_i p_i \log p_i = -1 \cdot \log 1 = 0 $$

so teacher forcing :

$$ \text{CE}(p, q) = D_{KL}(p \| q) $$

is exactlytraininglanguagemodel, CE loss KL divergence value. , one-hot supervised entropy result. 

In [ ]:
# valueverify: teacher forcing CE == KL
import torch
import torch.nn.functional as F

# vocabularysize 5, correct token id=2
p = torch.tensor([0.0, 0.0, 1.0, 0.0, 0.0]) # one-hot real
logits = torch.tensor([1.2, 0.5, 2.8, -0.3, 0.1])
q = F.softmax(logits, dim=-1) # modelpredict

# Cross-Entropy: -sum p_i * log q_i
log_q = torch.log(q)
ce = -(p * log_q).sum().item()

# entropy of p: -sum p_i * log p_i
# p id=2 1, log(1)=0, so H(p) = 0
entropy_p = -(p * torch.log(p.clamp_min(1e-12))).sum().item()

# KL(p || q) = -sum p_i * log(q_i / p_i)
kl = (p * (torch.log(p.clamp_min(1e-12)) - log_q)).sum().item()

print(f"H(p) = {entropy_p:.4f}")
print(f"CE(p, q) = {ce:.4f}")
print(f"KL(p||q) = {kl:.4f}")
print()
print(f"verify CE == H(p) + KL: {entropy_p:.4f} + {kl:.4f} = {entropy_p + kl:.4f}")
print("Key observation: teacher forcing real one-hot, H(p)=0, so CE KL value. ")

### 11.3 In

CE, KL, entropy Pre-training loss , Indifferenttraining, .

| training | | |
|:---|:---|:---|
| Pre-training / SFT | CE loss | supervised one-hot, CE = KL, |
| | KL divergence | teacher labels, one-hot, complete |
| RLHF (PPO) | KL penalty | In reward KL(policy \| reference), |
| DPO | log-ratio ( KL) | through log σ(β log(π/π_ref)) , equivalent KL |
| MoE load balancing | entropy / KL | router token expert, expert |

MoE load balancing loss aexample: each token every expert routerprobability $r$, $r$ In batch . entropy ( router entropy ) , KL($r$ \| uniform), .

: traininggoal"aa", CE, KL, entropy . teacher forcing ——one-hot supervisedvalue, soPre-training loss CE . 

## 12. A Common Confusion: Does the Model Shift, or Does the Data?

:

```python
input_ids = tokens[:-1]
labels = tokens[1:]
```

but Transformers modeltraining:

```python
input_ids = tokens
labels = tokens
```

. differenceIn **shift In**.

| | shift positions | |
|:---|:---|:---|
| `input=tokens[:-1]`, `labels=tokens[1:]` | data | , |
| `input=tokens`, `labels=tokens` | model loss shift | industrial, manually |

 `AutoModelForCausalLM` In loss , logits labels :

```text
shift_logits = logits[..., :-1, :]
shift_labels = labels[..., 1:]
```

sosee `labels=input_ids` , “modelpredict”. : currentpositionspredictnext token. 

In [ ]:
# compare: data shift model shift getsupervised
full_tokens = torch.tensor([[1, 3, 4, 5, 6, 2]])

external_input = full_tokens[:, :-1]
external_labels = full_tokens[:, 1:]

internal_input = full_tokens
internal_labels = full_tokens
shifted_labels_inside_model = internal_labels[:, 1:]

print(" shift:")
print("input: ", external_input.tolist())
print("labels:", external_labels.tolist())
print()

print(" shift equivalentresult:")
print("input complete tokens: ", internal_input.tolist())
print("model labels[:, 1:]:", shifted_labels_inside_model.tolist())
print()

same = torch.equal(external_labels, shifted_labels_inside_model)
print("predictgoal？", same)
print("Key observation: industrial shift Inmodel loss compute. ")

## Exercises

> can AI Explanation, step, check, but AI “”. 

**Exercise 1: construct SFT labels**

conversation token:

```text
tokens = ["<user>", "", "<assistant>", "", "", "<eos>"]
ids = [1, 10, 2, 10, 11, 3]
```

construct `labels`: assistant assistant reply `["", "", "<eos>"]` loss, the restpositions `-100`.

: assistant assistant reply id 10 , EOS id 3 . 

In [ ]:
# Exercise 1: construct SFT labels
ids = [1, 10, 2, 10, 11, 3]

# TODO: assistant assistant replypart label, the restpositions -100
labels = [-100, -100, -100, 10, 11, 3]

assert labels is not None, "construct labels"
assert labels == [-100, -100, -100, 10, 11, 3], f"labels , get {labels}"

print("✅ Exercise 1 through: ")
print(" user/template part mask , training only the assistant assistant reply. ")

**Exercise 2: computegradients batch size**

trainingparameter:

```text
per_device_train_batch_size = 2
num_gpus = 4
gradient_accumulation_steps = 8
```

compute optimizer step seesample.

: batch size = batch × GPU × gradients. 

In [ ]:
# Exercise 2: compute batch size
per_device_train_batch_size = 2
num_gpus = 4
gradient_accumulation_steps = 8

# TODO: compute optimizer.step() sample
effective_batch_size = per_device_train_batch_size * num_gpus * gradient_accumulation_steps

assert effective_batch_size is not None, "compute effective_batch_size"
assert effective_batch_size == 64, f"answer 64, get {effective_batch_size}"

print("✅ Exercise 2 through: ")
print(f" batch size = {effective_batch_size}")
print(" is exactlyVRAM gradient accumulation reason. ")

**Exercise 3: average token loss**

 batch 6 positions, eachpositions loss :

```text
losses = [0.2, 0.4, 1.0, 0.3, 0.8, 0.5]
labels = [10, 11, -100, 3, -100, 12]
```

computepositionsaverage loss. `labels == -100` positions.

: positions 0, 1, 3, 5. 

In [ ]:
# Exercise 3: average token loss
losses = [0.2, 0.4, 1.0, 0.3, 0.8, 0.5]
labels = [10, 11, -100, 3, -100, 12]

# TODO: filter labels == -100 positions, average
valid_average_loss = sum(loss for loss, label in zip(losses, labels) if label != -100) / sum(label != -100 for label in labels)

assert valid_average_loss is not None, "compute valid_average_loss"
expected = (0.2 + 0.4 + 0.3 + 0.5) / 4
assert abs(valid_average_loss - expected) < 1e-6, f"answer {expected:.4f}"

print("✅ Exercise 3 through: ")
print(f" average loss = {valid_average_loss:.4f}")
print(" ignore_index=-100 . ")

## Summary (Checklist)

:

1. ✅ `Trainer` / `ms-swift` It is not magic, training loop 
2. ✅ sample: → tokenize → construct labels → collator batch → model forward → loss
3. ✅ Causal LM goal next-token prediction
4. ✅ `logits` eachpositionsvocabulary, Cross-Entropy correct token probability
5. ✅ `-100` positions loss, PAD SFT user/system part
6. ✅ `attention_mask` model" pad", `labels=-100` loss" pad"
7. ✅ Hugging Face Transformers `Trainer` ModelScope `ms-swift` In
8. ✅ `gradient_accumulation_steps` updateparameter, forward batch shape
9. ✅ LoRA "parametertraining", loss/backward/optimizer step 
10. ✅ DeepSpeed/FSDP parameteroptimizerInmulti-GPU, traininggoal
11. ✅ Adam $m$ (momentum) $v$ (variance) , bias correction 
12. ✅ AdamW weight decay gradients; bias LayerNorm weight decay, parameter group 
13. ✅ Gradient clipping (global norm clip, $\\tau = 1.0$) update, scaling
14. ✅ Muon Newton-Schulz iterationgradientsmatrixorthogonalization, , , 2D parameteruse
15. ✅ AdamW modelPre-training, optimizerlearning rate

: industrialtraining, but `batch -> model -> loss -> backward -> step` , will not `trainer.train()` .

 Scaling Laws: training, nextis exactly"model, data, ". 

## Hands-On: From Pre-training to SFT

 Trainer construct token ExplanationTraining loop. Inreal, 
"Data cleaning → Pre-training (Pre-training, PT) → supervisedFine-tuning (Supervised Fine-Tuning, SFT) "pipeline.

training？PT modellearning"next token ", SFT 
learning"see user , assistant ". traininguse Cross-Entropy,
Indata token loss.

Hands-Onlayer. layer Notebook realrun demo: model 9.4M parameter, PT SFT
 150 step, "cleaning → training → generate"verify, language; layer
 llm_train/ full-scale 64M , 16 , 17 benchmark.

demo part:

1. DataJuicer cleaning 5,000 real BelleGroup conversation
2. 1 training Tokenizer, packing length 256 trainingblock
3. GQA qk_norm model, demo Pre-training 150 step
4. Chat Template assistant-only loss mask, SFT 150 step

### 13. Clean the Data with DataJuicer First

**DataJuicer**: data, 20+ ; In plain words, filter, deduplication
cleaningsteprepeatrunconfig. 2 , becausetrainingdatarepeat
model.

 `text_length_filter` `document_deduplicator`, data. 
 [data](15-data-engineering.ipynb).

 `/tmp/belle_sft.jsonl` 116 conversation 5,000 . 
completedata, rungetresult, cleaning. 

In [ ]:
# Tools for the hands-on section; these imports serve only the standalone practice below
import json
import subprocess
import sys
from pathlib import Path

import matplotlib.pyplot as plt
from tokenizers import Tokenizer

torch.manual_seed(42)

def find_project_root():
 """Walk up from cwd to find the project root and return it as a Path."""
 candidates = [Path.cwd(), *Path.cwd().parents]
 for candidate in candidates:
 if (candidate / "notebooks").is_dir() and (candidate / "llm_train").is_dir():
 return candidate
 raise FileNotFoundError("at the same time notebooks llm_train ")

project_root = find_project_root()
work_dir = Path("/tmp/modern_llm_station2")
work_dir.mkdir(parents=True, exist_ok=True)

sft_source_path = Path("/tmp/belle_sft.jsonl")
sample_path = work_dir / "belle_sft_5000.jsonl"
cleaned_path = work_dir / "belle_sft_5000_clean.jsonl"
config_path = work_dir / "datajuicer.yaml"

assert sft_source_path.exists(), " /tmp/belle_sft.jsonl, BelleGroup conversationdata"
print(f": {project_root}")
print(f"DataJuicer : {work_dir}")

In [ ]:
def clean_with_python(input_path, output_path):
 """Length filtering + exact dedup in pure Python; returns kept count."""
 seen = set()
 with input_path.open(encoding="utf-8") as source, output_path.open(
 "w", encoding="utf-8"
 ) as target:
 for line in source:
 item = json.loads(line)
 text = item["text"]
 if 20 <= len(text) <= 1200 and text not in seen:
 seen.add(text)
 target.write(json.dumps(item, ensure_ascii=False) + "\n")
 return len(seen)

print(" Python fallback : DataJuicer automaticexecutefilterdeduplication. ")

In [ ]:
# DataJuicer defaultread text , conversations at the same timea text
sample_count = 0
with sft_source_path.open(encoding="utf-8") as source:
 with sample_path.open("w", encoding="utf-8") as target:
 for line in source:
 item = json.loads(line)
 turns = item.get("conversations", [])
 item["text"] = "\n".join(turn.get("content", "") for turn in turns)
 target.write(json.dumps(item, ensure_ascii=False) + "\n")
 sample_count += 1
 if sample_count == 5000:
 break

config_text = f"""
project_name: station2_demo
dataset_path: '{sample_path}'
text_keys: text
np: 1
export_path: '{cleaned_path}'
process:
 - text_length_filter:
 min_len: 20
 max_len: 1200
 - document_deduplicator:
 lowercase: false
 ignore_non_character: false
""".strip()
config_path.write_text(config_text + "\n", encoding="utf-8")

datajuicer_used = False
try:
 command = [
 sys.executable,
 "-m",
 "data_juicer.tools.process_data",
 "--config",
 str(config_path),
 ]
 subprocess.run(command, check=True, capture_output=True, text=True)
 datajuicer_used = True
except (ModuleNotFoundError, subprocess.CalledProcessError) as error:
 clean_with_python(sample_path, cleaned_path)
 print(f"DataJuicer , Python fallback: {type(error).__name__}")

with cleaned_path.open(encoding="utf-8") as source:
 cleaned_count = sum(1 for _ in source)

runner_name = "DataJuicer" if datajuicer_used else " Python fallback"
print(f"execute: {runner_name}")
print(f"cleaning: {sample_count} ; cleaning: {cleaned_count} ")
print("Key observation: configvalue, cancleaning. ")

In [ ]:
# , countcan
fig, ax = plt.subplots(figsize=(5.5, 3.2))
bars = ax.bar(["Raw sample", "After cleaning"], [sample_count, cleaned_count])
ax.bar_label(bars)
ax.set_ylabel("Samples")
ax.set_title("Data Cleaning Result")
ax.set_ylim(0, sample_count * 1.12)
plt.show()

if DataJuicer, `clean_with_python` is exactlyequivalent fallback: read JSONL,
characterlength filtering, deduplication. 5,000 sample; datascale,
DataJuicer config, parallelexecute.

runway:

```bash
pip install py-data-juicer
python -m data_juicer.tools.process_data --config /tmp/modern_llm_station2/datajuicer.yaml
```

### 14. Pre-training Hands-On: Every Token in Continuous Text Is an Answer

**Pre-training (PT) **: modelIn next-token prediction, learninglanguage
structure. In plain words, eachpositionsnext token, modelanswer.

complete"64M teaching model" 8 layer, 768 hidden dim, 8 query head, 4 KV head, GQA,
qk_norm SwiGLU, parameter 62M. Notebook In CPU GPU ,
realrun 9.4M parameter; layer, traininggoal.
training demo: 9.4M, 150 step verifytraining, full-scale 64M In 16 .

layer: demo 1 `cn_corpus_sample.txt` ( 9 MB ) , load;
full-scale 64M Ultra-FineWeb-zh (cleaning 2.7 token) . layerdatascale
count, so demo verify", loss ", cannotpredictfull-scalebenchmarkscore.

| config | layer | hidden dim | query/KV head | FFN dim | parameter |
|:---|---:|---:|:---:|---:|---:|
| 64M teaching model | 8 | 768 | 8 / 4 | 2304 | 62M |
| Notebook | 4 | 384 | 6 / 2 | 1152 | 9.4M |

**GQA (Grouped-Query Attention) **: query head key/value head, KV
parameterinference. 6 query head 2 KV head, is exactly 3 query head K/V.

**qk_norm**: IncomputeNotescore, query key, value.
In plain words, "", . 

In [ ]:
# load 1 training 6400 vocab Tokenizer, packing block=256
tokenizer_path = project_root / "notebooks/part1-foundation/mini_tokenizer.json"
corpus_path = project_root / "notebooks/part1-foundation/data/cn_corpus_sample.txt"

assert tokenizer_path.exists(), "run 1 , generate mini_tokenizer.json"
assert corpus_path.exists(), "Pre-training cn_corpus_sample.txt"

tokenizer = Tokenizer.from_file(str(tokenizer_path))
vocab_size = tokenizer.get_vocab_size()
bos_id = tokenizer.token_to_id("<s>")
eos_id = tokenizer.token_to_id("</s>")
pad_id = tokenizer.token_to_id("<pad>")

corpus_text = corpus_path.read_text(encoding="utf-8")
token_ids = [bos_id] + tokenizer.encode(corpus_text).ids + [eos_id]
block_size = 256
num_blocks = len(token_ids) // (block_size + 1)
packed_ids = token_ids[:num_blocks * (block_size + 1)]
pt_blocks = torch.tensor(packed_ids).view(num_blocks, block_size + 1)

line_lengths = [len(tokenizer.encode(line).ids) for line in corpus_text.splitlines()[:500]]
print(f"vocabulary: {vocab_size}; token: {len(token_ids):,}")
print(f"After packing: {num_blocks:,} training blocks, each providing {block_size} prediction positions")
print("Decode the start of the first block: ", tokenizer.decode(pt_blocks[0, :40].tolist()))

fig, ax = plt.subplots(figsize=(6.5, 3.2))
ax.hist(line_lengths, bins=25, color="#4C78A8", edgecolor="white")
ax.axvline(block_size, color="#E45756", linestyle="--", label="Block size = 256")
ax.set_xlabel("Tokens per source line")
ax.set_ylabel("Lines")
ax.set_title("Source Lengths Before Packing")
ax.legend()
plt.show()

eachblock 257 token, `block_size=256`？because 1 256 token input,
 2 257 token labels:

```text
completeblock: [t0, t1, t2, ..., t255, t256]
input_ids: [t0, t1, t2, ..., t255]
labels: [t1, t2, t3, ..., t256]
```

Packing token , block. eachpositions loss, need
padding . realIn EOS, attention . 

In [ ]:
class RMSNorm(nn.Module):
 """finallyinput. """

 def __init__(self, hidden_size, eps=1e-6):
 """savelearningscalingparametervalue. """
 super().__init__()
 self.weight = nn.Parameter(torch.ones(hidden_size))
 self.eps = eps

 def forward(self, x):
 """returns shape result. """
 scale = torch.rsqrt(x.float().pow(2).mean(dim=-1, keepdim=True) + self.eps)
 return (x.float() * scale).to(x.dtype) * self.weight

def apply_rope(x):
 """Note head RoPE, model token order. """
 seq_len = x.size(-2)
 head_dim = x.size(-1)
 positions = torch.arange(seq_len, device=x.device, dtype=torch.float32)
 dimensions = torch.arange(0, head_dim, 2, device=x.device, dtype=torch.float32)
 frequencies = 1.0 / (10000 ** (dimensions / head_dim))
 angles = torch.outer(positions, frequencies)
 cos = angles.cos()[None, None, :, :].to(x.dtype)
 sin = angles.sin()[None, None, :, :].to(x.dtype)
 even = x[..., 0::2]
 odd = x[..., 1::2]
 rotated = torch.stack([even * cos - odd * sin, even * sin + odd * cos], dim=-1)
 return rotated.flatten(-2)

class GQAAttention(nn.Module):
 """ RoPE qk_norm Grouped-Query Attention. """

 def __init__(self, hidden_size, num_heads, num_kv_heads):
 """create Q/K/V/O , query KV head count. """
 super().__init__()
 assert hidden_size % num_heads == 0
 assert num_heads % num_kv_heads == 0
 self.num_heads = num_heads
 self.num_kv_heads = num_kv_heads
 self.head_dim = hidden_size // num_heads
 kv_size = num_kv_heads * self.head_dim
 self.q_proj = nn.Linear(hidden_size, hidden_size, bias=False)
 self.k_proj = nn.Linear(hidden_size, kv_size, bias=False)
 self.v_proj = nn.Linear(hidden_size, kv_size, bias=False)
 self.o_proj = nn.Linear(hidden_size, hidden_size, bias=False)

 def forward(self, x):
 """input [batch, time, hidden], returns shape Noteresult. """
 batch_size, seq_len, hidden_size = x.shape
 q = self.q_proj(x).view(
 batch_size, seq_len, self.num_heads, self.head_dim
 ).transpose(1, 2)
 k = self.k_proj(x).view(
 batch_size, seq_len, self.num_kv_heads, self.head_dim
 ).transpose(1, 2)
 v = self.v_proj(x).view(
 batch_size, seq_len, self.num_kv_heads, self.head_dim
 ).transpose(1, 2)

 q = apply_rope(q)
 k = apply_rope(k)
 q = q * torch.rsqrt(q.float().pow(2).mean(dim=-1, keepdim=True) + 1e-6)
 k = k * torch.rsqrt(k.float().pow(2).mean(dim=-1, keepdim=True) + 1e-6)
 q = q.to(x.dtype)
 k = k.to(x.dtype)

 repeats = self.num_heads // self.num_kv_heads
 k = k.repeat_interleave(repeats, dim=1)
 v = v.repeat_interleave(repeats, dim=1)
 attended = F.scaled_dot_product_attention(q, k, v, is_causal=True)
 attended = attended.transpose(1, 2).contiguous().view(
 batch_size, seq_len, hidden_size
 )
 return self.o_proj(attended)

In [ ]:
class SwiGLU(nn.Module):
 """, hidden size. """

 def __init__(self, hidden_size, ffn_size):
 """create gate, up down layer. """
 super().__init__()
 self.gate = nn.Linear(hidden_size, ffn_size, bias=False)
 self.up = nn.Linear(hidden_size, ffn_size, bias=False)
 self.down = nn.Linear(ffn_size, hidden_size, bias=False)

 def forward(self, x):
 """returns SwiLU result. """
 return self.down(F.silu(self.gate(x)) * self.up(x))

class TransformerBlock(nn.Module):
 """ GQA SwiGLU, use Pre-Norm . """

 def __init__(self, hidden_size, num_heads, num_kv_heads, ffn_size):
 """createa Transformer Block layer. """
 super().__init__()
 self.attn_norm = RMSNorm(hidden_size)
 self.attn = GQAAttention(hidden_size, num_heads, num_kv_heads)
 self.ffn_norm = RMSNorm(hidden_size)
 self.ffn = SwiGLU(hidden_size, ffn_size)

 def forward(self, x):
 """execute attention FFN. """
 x = x + self.attn(self.attn_norm(x))
 return x + self.ffn(self.ffn_norm(x))

class TeachingLM(nn.Module):
 """a PT SFT Decoder-only Causal Language Model. """

 def __init__(
 self,
 vocab_size,
 hidden_size,
 num_layers,
 num_heads,
 num_kv_heads,
 ffn_size,
 ):
 """configcreate Embedding, Transformer Blocks outputlayer. """
 super().__init__()
 self.token_embedding = nn.Embedding(vocab_size, hidden_size)
 self.blocks = nn.ModuleList([
 TransformerBlock(hidden_size, num_heads, num_kv_heads, ffn_size)
 for _ in range(num_layers)
 ])
 self.final_norm = RMSNorm(hidden_size)
 self.lm_head = nn.Linear(hidden_size, vocab_size, bias=False)
 self.lm_head.weight = self.token_embedding.weight
 self.apply(self._init_weights)

 def _init_weights(self, module):
 """initialize Linear Embedding. """
 if isinstance(module, (nn.Linear, nn.Embedding)):
 nn.init.normal_(module.weight, mean=0.0, std=0.02)

 def forward(self, input_ids, labels=None):
 """returns logits; labels returns token average CE loss. """
 hidden = self.token_embedding(input_ids)
 for block in self.blocks:
 hidden = block(hidden)
 logits = self.lm_head(self.final_norm(hidden))
 loss = None
 if labels is not None:
 loss = F.cross_entropy(
 logits.reshape(-1, logits.size(-1)),
 labels.reshape(-1),
 ignore_index=-100,
 )
 return logits, loss

In [ ]:
# demo: trains the compact model; the full-scale 64M config lives in llm_train/configs/firstllm_64m_exp24.yaml
full_config = {
 "hidden_size": 768,
 "num_layers": 8,
 "num_heads": 8,
 "num_kv_heads": 4,
 "ffn_size": 2304,
}
compact_config = {
 "hidden_size": 384,
 "num_layers": 4,
 "num_heads": 6,
 "num_kv_heads": 2,
 "ffn_size": 1152,
}

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = TeachingLM(vocab_size=vocab_size, **compact_config).to(device)
compact_parameters = sum(parameter.numel() for parameter in model.parameters())

def estimate_parameters(config, vocab_size):
 """weighttotal Decoder modelParameters. """
 hidden = config["hidden_size"]
 head_dim = hidden // config["num_heads"]
 kv_size = config["num_kv_heads"] * head_dim
 attention = hidden * hidden * 2 + hidden * kv_size * 2
 ffn = hidden * config["ffn_size"] * 3
 return vocab_size * hidden + config["num_layers"] * (attention + ffn)

full_parameters = estimate_parameters(full_config, vocab_size)
print(f"training: {device}")
print(f"realParameters: {compact_parameters / 1e6:.2f}M")
print(f"completeconfigParameters: {full_parameters / 1e6:.2f}M")

fig, ax = plt.subplots(figsize=(5.5, 3.2))
values = [compact_parameters / 1e6, full_parameters / 1e6]
bars = ax.bar(["Compact", "Full teaching model"], values, color=["#4C78A8", "#F58518"])
ax.bar_label(bars, fmt="%.1fM")
ax.set_ylabel("Parameters (millions)")
ax.set_title("Model Size Comparison")
ax.set_ylim(0, max(values) * 1.15)
plt.show()

#### 14.1 loss compute

cannotmodelstructure. a packed block , compute:

1. `input_ids = block[:-1]`, `labels = block[1:]`
2. modeleachpositionsoutput 6,400 logits
3. eachpositionscorrect token probability, finally 256 positions mean

at the same time `reduction="none"` modeldefault mean compute. . 

In [ ]:
check_x = pt_blocks[:1, :-1].to(device)
check_y = pt_blocks[:1, 1:].to(device)

model.eval()
with torch.no_grad():
 check_logits, model_loss = model(check_x, check_y)
 token_losses = F.cross_entropy(
 check_logits.reshape(-1, vocab_size),
 check_y.reshape(-1),
 reduction="none",
 )
manual_mean = token_losses.mean()

assert torch.allclose(manual_mean, model_loss, atol=1e-6)
print(f" token CE mean: {manual_mean.item():.6f}")
print(f"modelreturns loss: {model_loss.item():.6f}")
print("Key observation: labels shift each token CE average, is exactlyPre-training loss. ")

fig, ax = plt.subplots(figsize=(7, 3.2))
positions = list(range(20))
ax.bar(positions, token_losses[:20].float().cpu().tolist(), color="#54A24B")
ax.axhline(model_loss.item(), color="#E45756", linestyle="--", label="Block mean")
ax.set_xlabel("Token position")
ax.set_ylabel("Cross-entropy")
ax.set_title("Per-token Loss Before Pre-training")
ax.legend()
plt.show()

#### 14.2 Demo Pre-training for 150 Steps

training packed block. Notebook use AdamW, gradients.
 demo training: 150 step language, verifyreal (`cn_corpus_sample.txt`) ,
realmodelreal loss ; full-scale 64M trainingway Ultra-FineWeb 16 .

initialize, 6,400 token , loss $\ln(6400)\approx 8.76$. iftraining
correct, , but step becausedifferent. 

In [ ]:
model.train()
pt_optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=0.1)
pt_steps = 150
pt_batch_size = 12
pt_losses = []
pt_generator = torch.Generator().manual_seed(42)

for step in range(pt_steps):
 indices = torch.randint(len(pt_blocks), (pt_batch_size,), generator=pt_generator)
 batch = pt_blocks[indices]
 input_ids = batch[:, :-1].to(device)
 labels = batch[:, 1:].to(device)

 _, loss = model(input_ids, labels)
 loss.backward()
 torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
 pt_optimizer.step()
 pt_optimizer.zero_grad(set_to_none=True)
 pt_losses.append(loss.item())

 if step % 25 == 0 or step == pt_steps - 1:
 print(f"PT step {step + 1:03d}/{pt_steps} | loss = {loss.item():.4f}")

window = 10
pt_smooth = [
 sum(pt_losses[max(0, i - window + 1):i + 1]) / min(i + 1, window)
 for i in range(len(pt_losses))
]
print(f" 10 step average: {pt_smooth[9]:.4f}; 10 step average: {pt_smooth[-1]:.4f}")
print("Key observation: loss , average. ")

fig, ax = plt.subplots(figsize=(7, 3.6))
ax.plot(pt_losses, alpha=0.3, color="#4C78A8", label="Step loss")
ax.plot(pt_smooth, color="#E45756", linewidth=2, label="10-step mean")
ax.axhline(math.log(vocab_size), color="gray", linestyle="--", label="Random baseline")
ax.set_xlabel("Training step")
ax.set_ylabel("Cross-entropy")
ax.set_title("Pre-training Loss")
ax.legend()
plt.show()

### 15. SFT Hands-On: assistant loss

**SFT (Supervised Fine-Tuning) **: InanswertrainingPre-trainingmodel, modellearning
conversation. In plain words, PT "", SFT "".

conversationneed Chat Template:

```text
<s>user
？
assistant
. </s>
```

 2 "Chat Template loss". `user`, `assistant`
corresponding labels `-100`, `</s>`. modelseecompletecontext, but
assistant part. 

In [ ]:
def build_sft_example(item, tokenizer, block_size):
 """ user/assistant conversationconvert to input_ids assistant-only labels. """
 turns = item.get("conversations", [])
 if len(turns) < 2:
 return None
 user_text = turns[0].get("content", "").strip()
 assistant_text = turns[1].get("content", "").strip()
 if turns[0].get("role") != "user" or turns[1].get("role") != "assistant":
 return None

 prefix_text = f"user\n{user_text}\nassistant\n"
 prefix = [bos_id] + tokenizer.encode(prefix_text).ids
 if len(prefix) >= block_size:
 return None
 answer = tokenizer.encode(assistant_text).ids + [eos_id]
 answer = answer[:block_size + 1 - len(prefix)]
 if not answer:
 return None

 full = prefix + answer
 padding = [pad_id] * (block_size + 1 - len(full))
 input_ids = (full + padding)[:-1]
 labels = [-100] * (len(prefix) - 1) + answer
 labels += [-100] * (block_size - len(labels))
 return input_ids, labels

sft_examples = []
with cleaned_path.open(encoding="utf-8") as source:
 for line in source:
 example = build_sft_example(json.loads(line), tokenizer, block_size)
 if example is not None:
 sft_examples.append(example)

sft_inputs = torch.tensor([example[0] for example in sft_examples])
sft_labels = torch.tensor([example[1] for example in sft_examples])
supervised_tokens = (sft_labels != -100).sum(dim=1)

first_valid = sft_labels[0] != -100
first_answer_ids = sft_labels[0][first_valid].tolist()
print(f"trainingconversation: {len(sft_examples):,} ")
print(f"Average supervised tokens: {supervised_tokens.float().mean().item():.1f}")
print("supervised: ", tokenizer.decode(first_answer_ids))
print("Key observation: input_ids , labels assistant . ")

fig, ax = plt.subplots(figsize=(8, 1.8))
mask_preview = (sft_labels[:8, :80] != -100).float()
image = ax.imshow(mask_preview, aspect="auto", cmap="Blues", vmin=0, vmax=1)
ax.set_xlabel("Token position")
ax.set_ylabel("Conversation")
ax.set_title("Assistant-only Loss Mask (First 80 Tokens)")
colorbar = fig.colorbar(image, ax=ax, pad=0.02)
colorbar.set_label("Supervised (1=yes)")
plt.show()

#### 15.1 demo SFT: PT weighttraining 150 step

createmodel. `model` PT parameter, data mask conversation,
learning rate `1e-4`. is exactly"model".

Note SFT loss In assistant token average, so PT loss rangedifferent. can
, butcannotvaluetraining. 

In [ ]:
model.train()
sft_optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=0.01)
sft_steps = 150
sft_batch_size = 12
sft_losses = []
sft_generator = torch.Generator().manual_seed(7)

for step in range(sft_steps):
 indices = torch.randint(len(sft_inputs), (sft_batch_size,), generator=sft_generator)
 input_ids = sft_inputs[indices].to(device)
 labels = sft_labels[indices].to(device)

 _, loss = model(input_ids, labels)
 loss.backward()
 torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
 sft_optimizer.step()
 sft_optimizer.zero_grad(set_to_none=True)
 sft_losses.append(loss.item())

 if step % 25 == 0 or step == sft_steps - 1:
 print(f"SFT step {step + 1:03d}/{sft_steps} | loss = {loss.item():.4f}")

sft_smooth = [
 sum(sft_losses[max(0, i - window + 1):i + 1]) / min(i + 1, window)
 for i in range(len(sft_losses))
]
print(f" 10 step average: {sft_smooth[9]:.4f}; 10 step average: {sft_smooth[-1]:.4f}")
print("Key observation: modelparameter PT, SFT data loss mask. ")

fig, ax = plt.subplots(figsize=(7, 3.6))
ax.plot(sft_losses, alpha=0.3, color="#72B7B2", label="Step loss")
ax.plot(sft_smooth, color="#F58518", linewidth=2, label="10-step mean")
ax.set_xlabel("Training step")
ax.set_ylabel("Masked cross-entropy")
ax.set_title("Supervised Fine-tuning Loss")
ax.legend()
plt.show()

#### 15.2 generate, observe 150 step 

**generate (generation) **: modelpredict token input, predictnext token, EOS 
length. answersampling, outputtrainingmodel.

 9.4M parameter, MB PT 150 step SFT , .
assistant reply, repeatneedrealresult. 

In [ ]:
def generate_answer(model, question, max_new_tokens=60):
 """training Chat Template generate assistant . """
 prefix = [bos_id] + tokenizer.encode(f"user\n{question}\nassistant\n").ids
 generated = []
 current = torch.tensor([prefix], device=device)
 model.eval()
 with torch.no_grad():
 for _ in range(max_new_tokens):
 logits, _ = model(current[:, -block_size:])
 next_logits = logits[:, -1, :] / 0.8
 top_values, top_indices = torch.topk(next_logits, k=40, dim=-1)
 probabilities = torch.softmax(top_values, dim=-1)
 sampled = torch.multinomial(probabilities, num_samples=1)
 next_id = top_indices.gather(-1, sampled).item()
 if next_id == eos_id:
 break
 generated.append(next_id)
 next_token = torch.tensor([[next_id]], device=device)
 current = torch.cat([current, next_token], dim=1)
 return tokenizer.decode(generated).strip()

torch.manual_seed(42)
demo_questions = ["Explanation. ", ", ？"]
fresh_generations = []
for question in demo_questions:
 answer = generate_answer(model, question)
 fresh_generations.append(answer)
 shown_answer = answer if answer else "[modelgenerate EOS]"
 print(f"user: {question}")
 print(f"assistant: {shown_answer}\n")

print("Key observation: trainingverifypipeline, cannotPre-trainingbenchmark. ")

#### 15.3 complete 64M 

"Notebook 150 step""modelscale", complete 64M training
generate. useclass GQA/qk_norm modelreal BelleGroup SFT data.
content[ 2 execute](../../llm_train/reports/station2_pt_sft_report.md).

****

```text
user: : ？
assistant: throughcomputeclassclass.
```

****

```text
user: , ……, 
assistant: , In, ……
 , ……
```

 loss , generate？Teacher forcing training"correctpredictnext
token"; generate, model token, wrong. 64M 10M 
model, , cannot. 3 unified,
repeatmetric lm-eval benchmark. 

### 16. Full-Scale 64M Reproduction Entry Point

demo verifypipeline. full-scalecompleteIn llm_train/ : config
, "Data cleaning → Pre-training → SFT → PPL → lm-eval", modelstructure
Notebook , block 256 512.

dataNote: full-scalePre-trainingdata Ultra-FineWeb-zh (Data-Juicer 1.5.5 cleaning) ,
 MiniMind data, MiniMind . cleaning score (mini 
score>=0.8, full score>=0.7) , content mapper length filtering 100-8000 character; SimHash
deduplication, becauseIn 57% , repeat 0.1%.
token budgetreal BPE truncation: mini 4 2.7 token (216,186 ) , full 
20 goal 22.1 . 

In [ ]:
# Load the full-scale 64M config, check llm_train/ assets, and print reproduction commands
import yaml

config_path = project_root / "llm_train/configs/firstllm_64m_exp24.yaml"
asset_names = [
 "llm_train/configs/firstllm_64m_exp24.yaml",
 "llm_train/modeling_firstllm.py",
 "llm_train/train_pretrain.py",
 "llm_train/train_sft.py",
 "llm_train/preprocess_ufw.py",
 "llm_train/scripts/run_ppl.py",
 "llm_train/scripts/run_lm_eval.sh",
]
missing = [name for name in asset_names if not (project_root / name).exists()]
assert not missing, f": {missing}"

with config_path.open(encoding="utf-8") as source:
 firstllm_cfg = yaml.safe_load(source)
model_cfg = firstllm_cfg["model"]
train_cfg = firstllm_cfg["train"]
print(f": {model_cfg['num_layers']} layer / hidden {model_cfg['hidden_size']} / "
 f"Q{model_cfg['num_query_heads']} KV{model_cfg['num_kv_heads']} / qk_norm")
print(f"training: block {firstllm_cfg['data']['block_size']} x batch "
 f"{train_cfg['batch_size']} = 64K token/step")
print(f"learning rate: mini {train_cfg['lr_mini']} / full {train_cfg['lr_full']}; "
 f"warmup {train_cfg['warmup_steps']}; seed {firstllm_cfg['run']['seeds']}")

launch_commands = [
 "python llm_train/preprocess_ufw.py download --tier mini --raw-dir data/ufw_raw",
 "python llm_train/preprocess_ufw.py clean --tier mini --raw-dir data/ufw_raw"
 " --work-dir data/ufw_mini",
 "python llm_train/preprocess_ufw.py truncate --tier mini --cleaned"
 " data/ufw_mini/ufw_mini_clean.jsonl --out data/ufw_mini",
 "python llm_train/preprocess_ufw.py pack --tier mini --final"
 " data/ufw_mini/ufw_mini.jsonl --out data/ufw_mini"
 " --tokenizer notebooks/part1-foundation/mini_tokenizer.json",
 "python llm_train/train_pretrain.py --config llm_train/configs/"
 "firstllm_64m_exp24.yaml --tier mini --seed 42",
 "python llm_train/train_sft.py --config llm_train/configs/"
 "firstllm_64m_exp24.yaml --ckpt llm_train/checkpoints/firstllm_64m_exp24/"
 "mini_seed42.pt",
 "bash llm_train/scripts/run_lm_eval.sh # PPL : scripts/run_ppl.py",
]
print("\n (orderexecute, full --tier full) : ")
for command in launch_commands:
 print(" ", command)
print("Key observation: config, yaml i.e. mini/full seed. ")

### 17. Historical Real Results (Historical)

 FirstLLM trainingresult (2026-08-21 update) , ,
** Notebook run**. everyusea 64M (Dense GQA + qk_norm +
SwiGLU) 6400 vocabulary; custom training tokenizer, MM MiniMind data
.

| | tokenizer | PT data | ceval | cmmlu | mmlu | gsm8k | piqa | wino |
|:---|:---|---:|---:|---:|---:|---:|---:|---:|
| custom 2.4B | training 6400 | 2.4B | 25.0 | 25.4 | 23.2 | 0.6 | 55.4 | 49.9 |
| custom 3.1B | training 6400 | 3.1B | 22.4 | 25.1 | 24.5 | 0.6 | 55.0 | 51.2 |
| custom 4.8B | training 6400 | 4.8B | 23.3 | 24.8 | 23.2 | 0.6 | 53.4 | 52.3 |
| custom 10B | training 6400 | 10B | 23.3 | 25.1 | 23.3 | 0.6 | 55.1 | 50.8 |
| MM s777 | MM 6400 | 2.39B | 25.7 | 24.8 | 37.4* | 0.5 | 54.5 | 50.8 |
| MM sky100_v4 | MM 6400 | 36.4B | 26.4 | 25.3 | 25.0 | 0.6 | 53.3 | 51.4 |
| MM skyfull | MM 6400 | 31.6B | 25.9 | 24.9 | 25.9 | 0.6 | 53.9 | 48.9 |

*mmlu 37.4 loglikelihood evaluation, 64M model.

:

1. 64M modelIn 2B token : 10B 2.4B ceval (23.3)
2. training tokenizer MM: 2.4B ceval 25.0 25.7, 0.7
3. vocabulary 12800 : embedding parameter, transformer 
4. gsm8k In 0.5-0.6%: 64M scaleinference, data

In [ ]:
# Historical chart: ceval vs PT data budget (data from the historical table above)
custom_tokens = [2.4, 3.1, 4.8, 10.0]
custom_ceval = [25.0, 22.4, 23.3, 23.3]
mm_tokens = [2.39, 31.6, 36.4]
mm_ceval = [25.7, 25.9, 26.4]

fig, ax = plt.subplots(figsize=(6.8, 3.4))
ax.plot(custom_tokens, custom_ceval, marker="o", color="#4C78A8",
 label="Self-trained tokenizer")
ax.plot(mm_tokens, mm_ceval, marker="s", color="#F58518", label="MiniMind baseline")
ax.axvline(2.21, color="gray", linestyle="--", label="MiniMind full budget (2.21B)")
ax.set_xlabel("Pre-training tokens (billions)")
ax.set_ylabel("CEval score")
ax.set_title("Historical 64M Results: CEVal vs Data Budget")
ax.legend(fontsize=8)
plt.show()

print("Key observation: 2.4B ceval , 64M modeldatabudget. ")
print("In 2.4B , Training tokensizer 25.0 MM 25.7. ")

### Hands-On Summary (Checklist)

Explanation:

1. ✅ DataJuicer configlength filteringdeduplication; Python fallback
2. ✅ PT token packing lengthblock, every labels shift 
3. ✅ 64M teaching modeluse 8 layer, 768 dim, GQA qk_norm; Notebook run
4. ✅ each token Cross-Entropy mean, is exactly PT loss
5. ✅ SFT PT weight, initializeamodel
6. ✅ Chat Template , `-100` user part SFT loss
7. ✅ loss NotetraininggoalIn, modelgeneratecorrect
8. ✅ demo full-scalelayer: full-scale 64M config, databenchmarkIn llm_train/
9. ✅ 17 benchmark (historical) , Notebook 

### Next station

> 🔬 **From-0 Hands-On 2/3 → 3/3 In
> [28-evaluation](../part5-production/28-evaluation.ipynb) (lm-eval benchmark) **
>
> Next station""repeatmetric, check. 